#### OASIS Re-Identification Experiment with Patient IDs

This is an upper-boundary stress test for patient re-identifiability from the MRI images before and after GS transformations. 

### Config

In [1]:
# =============================================================================
# 03_OASIS_reid_PIDs.ipynb
# Re-identification with Patient IDs (Slice-level split)
#
# CLEAN VERSION — GitHub publishable
# Logic preserved from original framework
#
# IMPORTANT:
# - This notebook uses PRECOMPUTED slice-level splits from 01_data_prep.ipynb
# - No data is re-split here
# - This is a SINGLE-SPLIT experiment (fold-0 equivalent), NOT true k-fold CV
# - Multiple repeats (NR) are used for statistics
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

# -----------------------------------------------------------------------------
# Reproducibility utilities
# -----------------------------------------------------------------------------

def set_global_seed(seed: int) -> None:
    """
    Set all relevant RNG seeds.
    NOTE:
    - Original framework used fixed seeds per repeat.
    - GS randomness is allowed upstream and not controlled here.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------------------------
# Project root & path resolution
# -----------------------------------------------------------------------------

# Assumes notebook is run from repo root (recommended for GitHub users)
PROJECT_ROOT = Path.cwd()

DATA_ROOT    = PROJECT_ROOT / "data"
MODELS_ROOT  = PROJECT_ROOT / "models"
RESULTS_ROOT = PROJECT_ROOT / "results"
CACHE_ROOT   = PROJECT_ROOT / "cache"

# Dataset-specific roots
OASIS_DATA_ROOT    = DATA_ROOT / "oasis"
OASIS_MODELS_ROOT  = MODELS_ROOT / "oasis" / "reid"
OASIS_RESULTS_ROOT = RESULTS_ROOT / "oasis" / "reid"
OASIS_CACHE_ROOT   = CACHE_ROOT / "oasis" / "reid"

# Processed data (from 01_data_prep.ipynb)
OASIS_REID_DATA_ROOT = OASIS_DATA_ROOT / "processed" / "reid"

# Create output directories (safe, no overwrite)
OASIS_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
OASIS_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
OASIS_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("📁 Project paths")
print(f"  Data (reid)   → {OASIS_REID_DATA_ROOT}")
print(f"  Models        → {OASIS_MODELS_ROOT}")
print(f"  Results       → {OASIS_RESULTS_ROOT}")
print(f"  Cache         → {OASIS_CACHE_ROOT}")


# -------------------------
# Experiment configuration 
# -------------------------

# Architectures evaluated
ARCHITECTURES = [
    "resnet18",
    "densenet121",
]

# Initialization modes
INITIALIZATIONS = [
    "Pretrained",
    "Scratch",
]

# GS conditions (must match saved results exactly)
GS_LEVELS = [
    "Raw",
    "GS 0%",
    "GS 10%",
    "GS 20%",
    "GS 30%",
    "GS 40%",
    "GS 50%",
]

GS_PERCENTAGE_MAP = {
    "Raw":   0.0,
    "GS 0%": 0.0,
    "GS 10%": 10.0,
    "GS 20%": 20.0,
    "GS 30%": 30.0,
    "GS 40%": 40.0,
    "GS 50%": 50.0,
}

GLOBAL_SEED = 1337

# Training fraction
TRAINING_FRACTION = 1.0 

# Number of independent repeats (NR)
NUM_REPEATS = 5  # NR5 in CSV

# Optimization hyperparameters
NUM_EPOCHS = 30
LEARNING_RATE = 5e-4
BATCH_SIZE = 16
GS_ITERATIONS = 50  # GSit50 in CSV

RESUME_IF_EXISTS = True

# -----------------------------------------------------------------------------
# Cache configuration
# -----------------------------------------------------------------------------

# Cache is OPTIONAL and OFF by default
# If enabled, fold-specific tensors will be saved under cache/oasis/reid
CACHE_ENABLED = False

print("\n⚙️ Experiment configuration")
print(f"  Architectures:        {ARCHITECTURES}")
print(f"  Initializations:      {INITIALIZATIONS}")
print(f"  GS levels:            {GS_LEVELS}")
print(f"  Training fraction:    {TRAINING_FRACTION}")
print(f"  Repeats (NR):         {NUM_REPEATS}")
print(f"  Epochs:               {NUM_EPOCHS}")
print(f"  Batch size:           {BATCH_SIZE}")
print(f"  Learning rate:        {LEARNING_RATE}")
print(f"  GS iterations:        {GS_ITERATIONS}")
print(f"  Cache enabled:        {CACHE_ENABLED}")


# -----------------------------------------------------------------------------
# Sanity checks (lightweight, non-destructive)
# -----------------------------------------------------------------------------

assert OASIS_REID_DATA_ROOT.exists(), (
    "Processed OASIS ReID data not found. "
    "Run 01_data_prep.ipynb first."
)

# NOTE:
# We do NOT check for specific files here yet.
# That will be done lazily at load time to avoid hard-coding filenames.

print("\n✅ Cell 1 complete. Environment and configuration initialized.")


📁 Project paths
  Data (reid)   → /home/jupyter/notebooks/reid_clean/data/oasis/processed/reid
  Models        → /home/jupyter/notebooks/reid_clean/models/oasis/reid
  Results       → /home/jupyter/notebooks/reid_clean/results/oasis/reid
  Cache         → /home/jupyter/notebooks/reid_clean/cache/oasis/reid

⚙️ Experiment configuration
  Architectures:        ['resnet18', 'densenet121']
  Initializations:      ['Pretrained', 'Scratch']
  GS levels:            ['Raw', 'GS 0%', 'GS 10%', 'GS 20%', 'GS 30%', 'GS 40%', 'GS 50%']
  Training fraction:    1.0
  Repeats (NR):         5
  Epochs:               30
  Batch size:           16
  Learning rate:        0.0005
  GS iterations:        50
  Cache enabled:        False

✅ Cell 1 complete. Environment and configuration initialized.


### Imports and Functions

In [2]:
# =============================================================================
# CELL 2: Imports & Core Utilities
# =============================================================================

import math
import pickle
from dataclasses import dataclass
from datetime import datetime

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import torchvision.transforms.functional as TF

from sklearn.metrics import confusion_matrix, f1_score, top_k_accuracy_score
from scipy import stats
from scipy.stats import entropy as scipy_entropy
from sklearn.metrics import mutual_info_score

# LPIPS (optional but recommended; original framework used it)
try:
    import lpips
    LPIPS_AVAILABLE = True
except Exception:
    LPIPS_AVAILABLE = False

# GS transform
from gs_functions import GS_batch_image


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🧠 Device: {DEVICE}")

# Fixed visualization/sample index (used for LPIPS/MI/Entropy comparability)
VIZ_SAMPLE_IDX = 0  # consistent with your original framework pattern


def seed_worker(worker_id: int) -> None:
    """Deterministic worker init for DataLoaders."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# LPIPS init (lazy + safe)
_lpips_model = None
def get_lpips_model():
    global _lpips_model
    if not LPIPS_AVAILABLE:
        return None
    if _lpips_model is None:
        _lpips_model = lpips.LPIPS(net="vgg").to(DEVICE).eval()
    return _lpips_model


print("✅ Cell 2 complete.")


🧠 Device: cuda
✅ Cell 2 complete.


In [3]:
# =============================================================================
# CELL 3: Data Loading (from data/oasis/processed/reid)
# =============================================================================

def _find_first_existing(root: Path, candidates: List[str]) -> Path:
    for c in candidates:
        p = root / c
        if p.exists():
            return p
    raise FileNotFoundError(
        f"None of these files exist under {root}:\n" + "\n".join([f"  - {x}" for x in candidates])
    )


def load_split_dicts(reid_root: Path) -> Tuple[List[dict], List[dict], List[dict]]:
    """
    Loads train/val/test from processed reid root.
    Supports either:
      - train.pkl / val.pkl / test.pkl (list[dict])
      - train.npz / val.npz / test.npz (arrays)
    """
    # Prefer pickle (closest to your original notebook)
    try:
        train_p = _find_first_existing(reid_root, ["train.pkl", "train_reid.pkl"])
        val_p   = _find_first_existing(reid_root, ["val.pkl", "val_reid.pkl"])
        test_p  = _find_first_existing(reid_root, ["test.pkl", "test_reid.pkl"])

        with open(train_p, "rb") as f: train_d = pickle.load(f)
        with open(val_p, "rb") as f:   val_d = pickle.load(f)
        with open(test_p, "rb") as f:  test_d = pickle.load(f)

        return train_d, val_d, test_d

    except FileNotFoundError:
        # Try NPZ fallback
        train_p = _find_first_existing(reid_root, ["train.npz"])
        val_p   = _find_first_existing(reid_root, ["val.npz"])
        test_p  = _find_first_existing(reid_root, ["test.npz"])

        def npz_to_dict_list(npz_path: Path) -> List[dict]:
            z = np.load(npz_path, allow_pickle=True)
            # expected keys: images, labels (pid)
            imgs = z["images"]
            lbls = z["labels"]
            out = [{"image": imgs[i], "reid_label": lbls[i]} for i in range(len(imgs))]
            return out

        return npz_to_dict_list(train_p), npz_to_dict_list(val_p), npz_to_dict_list(test_p)


def enforce_zero_one(x: np.ndarray, name: str) -> np.ndarray:
    xmin, xmax = float(x.min()), float(x.max())
    if xmin < 0.0 or xmax > 1.0:
        print(f"  🔧 {name}: range [{xmin:.3f},{xmax:.3f}] -> scaling to [0,1]")
        x = (x - xmin) / (xmax - xmin + 1e-8)
    return x


print("\n" + "="*80)
print("📦 Loading OASIS ReID processed splits")
print("="*80)

train_dicts, val_dicts, test_dicts = load_split_dicts(OASIS_REID_DATA_ROOT)

x_train = np.asarray([d["image"] for d in train_dicts])
x_val   = np.asarray([d["image"] for d in val_dicts])
x_test  = np.asarray([d["image"] for d in test_dicts])

pid_train_raw = np.asarray([d["reid_label"] for d in train_dicts])
pid_val_raw   = np.asarray([d["reid_label"] for d in val_dicts])
pid_test_raw  = np.asarray([d["reid_label"] for d in test_dicts])

print(f"Counts -> Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(x_test)}")

# Global encoding (critical)
all_pids = sorted(set(pid_train_raw) | set(pid_val_raw) | set(pid_test_raw))
pid_to_idx = {pid: i for i, pid in enumerate(all_pids)}
NUM_CLASSES = len(all_pids)

y_train = np.asarray([pid_to_idx[p] for p in pid_train_raw], dtype=np.int64)
y_val   = np.asarray([pid_to_idx[p] for p in pid_val_raw], dtype=np.int64)
y_test  = np.asarray([pid_to_idx[p] for p in pid_test_raw], dtype=np.int64)

print(f"Global PID classes: {NUM_CLASSES}")

# Normalize to [0,1] if needed (matches your original enforcement)
x_train = enforce_zero_one(x_train, "Train")
x_val   = enforce_zero_one(x_val, "Val")
x_test  = enforce_zero_one(x_test, "Test")

assert len(np.unique(np.concatenate([y_train, y_val, y_test]))) == NUM_CLASSES
print("✅ Label encoding verification passed.")
print("✅ Cell 3 complete.")



📦 Loading OASIS ReID processed splits
Counts -> Train: 1388 | Val: 347 | Test: 1041
Global PID classes: 347
✅ Label encoding verification passed.
✅ Cell 3 complete.


In [4]:
# =============================================================================
# CELL 4: Dataset & Model Factory
# =============================================================================

class AdaptiveNormDataset(Dataset):
    def __init__(self, imgs: np.ndarray, lbls: np.ndarray, mean: float, std: float, augment: bool):
        self.imgs = imgs
        self.lbls = lbls
        self.mean = float(mean)
        self.std  = float(std)
        self.augment = bool(augment)

    def __len__(self) -> int:
        return int(len(self.imgs))

    def __getitem__(self, idx: int):
        img = self.imgs[idx].copy()

        if self.augment:
            # Small augmentations consistent with original style
            if random.random() > 0.5:
                angle = random.uniform(-5, 5)
                img = TF.rotate(torch.tensor(img).unsqueeze(0), angle).numpy()[0]
            if random.random() > 0.5:
                img = np.fliplr(img)
            if random.random() > 0.5:
                bf = random.uniform(0.8, 1.2)
                lo, hi = img.min(), img.max()
                img = np.clip(img * bf, lo, hi)

        img = (img - self.mean) / (self.std + 1e-8)

        x = torch.tensor(img).unsqueeze(0).float()
        y = torch.tensor(int(self.lbls[idx]), dtype=torch.long)
        return x, y


def adapt_conv1_luminosity(old_conv: nn.Conv2d) -> nn.Conv2d:
    new_conv = nn.Conv2d(
        1, old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    with torch.no_grad():
        w = old_conv.weight.data.cpu()  # [out, 3, k, k]
        rgb = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
        new_conv.weight.data = (w * rgb).sum(dim=1, keepdim=True)
    return new_conv


def get_model(arch: str, init: str, num_classes: int) -> nn.Module:
    """
    init: 'Pretrained' or 'Scratch'
    """
    pretrained = (init.lower() == "pretrained")

    if arch == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.resnet18(weights=weights)
        m.conv1 = adapt_conv1_luminosity(m.conv1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m.to(DEVICE)

    if arch == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.densenet121(weights=weights)
        old = m.features[0]  # conv0
        new = nn.Conv2d(
            1, old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
            bias=False,
        )
        with torch.no_grad():
            w = old.weight.data.cpu()
            if pretrained:
                rgb = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
                new.weight.data = (w * rgb).sum(dim=1, keepdim=True)
            else:
                # scratch init handled by torchvision already; keep random init
                pass
        m.features[0] = new
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        return m.to(DEVICE)

    raise ValueError(f"Unknown architecture: {arch}")


print("✅ Cell 4 complete.")


✅ Cell 4 complete.


In [5]:
# =============================================================================
# CELL 5: Metrics (including info-theoretic + LPIPS)
# =============================================================================

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> Dict[str, object]:
    model.eval()
    preds, labels, probs = [], [], []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        p = torch.softmax(logits, dim=1)

        preds.append(logits.argmax(dim=1).cpu().numpy())
        labels.append(y.cpu().numpy())
        probs.append(p.cpu().numpy())

    preds = np.concatenate(preds)
    labels = np.concatenate(labels)
    probs = np.concatenate(probs)

    top1 = 100.0 * float(np.mean(preds == labels))

    k = min(5, probs.shape[1])
    if k > 1:
        top5 = 100.0 * float(top_k_accuracy_score(labels, probs, k=k))
    else:
        top5 = top1

    f1m = float(f1_score(labels, preds, average="macro", zero_division=0))

    # TrueProbTop5 (same logic as your original: prob assigned to true label if in top5 else 0)
    true_prob_top5 = []
    for i in range(len(labels)):
        top_idx = np.argsort(probs[i])[-5:][::-1]
        true_lbl = labels[i]
        if true_lbl in top_idx:
            true_prob_top5.append(float(probs[i, true_lbl]))
        else:
            true_prob_top5.append(0.0)
    true_prob_top5 = np.asarray(true_prob_top5, dtype=float)

    return dict(
        top1_acc=top1,
        top5_acc=top5,
        f1_macro=f1m,
        trueprob_top5_mean=float(true_prob_top5.mean()),
        trueprob_top5_std=float(true_prob_top5.std(ddof=1)) if len(true_prob_top5) > 1 else 0.0,
        trueprob_top5_median=float(np.median(true_prob_top5)),
    )


def _to_lpips_tensor(img2d: np.ndarray) -> torch.Tensor:
    # LPIPS expects [-1,1] and 3-channel
    t = torch.tensor(img2d).float().unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)
    return (t * 2.0) - 1.0


def compute_mutual_information(img1: np.ndarray, img2: np.ndarray, bins: int = 256) -> float:
    hist2d, _, _ = np.histogram2d(img1.flatten(), img2.flatten(), bins=bins)
    mi = mutual_info_score(None, None, contingency=hist2d)

    hist1, _ = np.histogram(img1.flatten(), bins=bins, density=True)
    hist2, _ = np.histogram(img2.flatten(), bins=bins, density=True)
    h1 = scipy_entropy(hist1 + 1e-10)
    h2 = scipy_entropy(hist2 + 1e-10)

    nmi = mi / (((h1 + h2) / 2) + 1e-10)
    return float(nmi)


def compute_image_entropy(img: np.ndarray, bins: int = 256) -> float:
    hist, _ = np.histogram(img.flatten(), bins=bins, density=True)
    return float(scipy_entropy(hist + 1e-10))


@torch.no_grad()
def compute_pair_metrics(raw_img: np.ndarray, alt_img: np.ndarray) -> Dict[str, float]:
    # Entropies (always computed)
    out = {
        "MutualInfo": compute_mutual_information(raw_img, alt_img),
        "Entropy_GS": compute_image_entropy(alt_img),
        "Entropy_Raw": compute_image_entropy(raw_img),
    }

    # LPIPS (optional)
    if LPIPS_AVAILABLE:
        m = get_lpips_model()
        v = m(_to_lpips_tensor(raw_img).to(DEVICE), _to_lpips_tensor(alt_img).to(DEVICE)).item()
        out["LPIPS"] = float(v)
    else:
        out["LPIPS"] = float("nan")

    return out


print("✅ Cell 5 complete.")


✅ Cell 5 complete.


In [6]:
# =============================================================================
# CELL 6: Cache Helpers (optional)
# =============================================================================

def _mask_to_key(gs_level: str) -> str:
    # "GS 10%" -> "10", "Raw" -> "raw"
    if gs_level == "Raw":
        return "raw"
    return str(int(round(GS_PERCENTAGE_MAP[gs_level])))

def _cache_dir_for(gs_level: str) -> Path:
    return OASIS_CACHE_ROOT / f"gs_it{GS_ITERATIONS}" / f"mask{_mask_to_key(gs_level)}"

def _cache_paths(gs_level: str) -> Dict[str, Path]:
    d = _cache_dir_for(gs_level)
    return dict(
        dir=d,
        train=d / "train.npy",
        val=d / "val.npy",
        test=d / "test.npy",
        meanstd=d / "meanstd.json",
    )

def load_or_build_condition_arrays(gs_level: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, float, float, Dict[str, float]]:
    """
    Returns: (x_tr, x_va, x_te, mean, std, pair_metrics)
    pair_metrics computed using VIZ_SAMPLE_IDX on raw test image vs condition test image.
    """
    if gs_level == "Raw":
        xtr, xva, xte = x_train, x_val, x_test
        mean = float(np.mean(np.concatenate([xtr, xva], axis=0)))
        std  = float(np.std(np.concatenate([xtr, xva], axis=0)))
        pm = {"LPIPS": 0.0, "MutualInfo": 1.0, "Entropy_GS": compute_image_entropy(x_test[VIZ_SAMPLE_IDX]), "Entropy_Raw": compute_image_entropy(x_test[VIZ_SAMPLE_IDX])}
        return xtr, xva, xte, mean, std, pm

    # GS transform
    cp = _cache_paths(gs_level)
    if CACHE_ENABLED and cp["train"].exists() and cp["val"].exists() and cp["test"].exists() and cp["meanstd"].exists():
        xtr = np.load(cp["train"])
        xva = np.load(cp["val"])
        xte = np.load(cp["test"])
        ms = json.loads(cp["meanstd"].read_text())
        mean, std = float(ms["mean"]), float(ms["std"])
        pm = ms["pair_metrics"]
        return xtr, xva, xte, mean, std, pm

    # Build fresh
    pct = GS_PERCENTAGE_MAP[gs_level] / 100.0
    xtr = GS_batch_image(x_train, batch_size=16, ite=GS_ITERATIONS, maskP=pct)
    xva = GS_batch_image(x_val, batch_size=16, ite=GS_ITERATIONS, maskP=pct)
    xte = GS_batch_image(x_test, batch_size=16, ite=GS_ITERATIONS, maskP=pct)

    # mean/std computed over (train+val) to match original “pool” idea
    pool = np.concatenate([xtr, xva], axis=0)
    mean = float(pool.mean())
    std  = float(pool.std())

    pm = compute_pair_metrics(x_test[VIZ_SAMPLE_IDX], xte[VIZ_SAMPLE_IDX])

    if CACHE_ENABLED:
        cp["dir"].mkdir(parents=True, exist_ok=True)
        np.save(cp["train"], xtr)
        np.save(cp["val"], xva)
        np.save(cp["test"], xte)
        cp["meanstd"].write_text(json.dumps({"mean": mean, "std": std, "pair_metrics": pm}, indent=2))

    return xtr, xva, xte, mean, std, pm


print("✅ Cell 6 complete.")


✅ Cell 6 complete.


In [7]:
# =============================================================================
# CELL 7: Training Engine (single split)
# =============================================================================

def train_one_repeat(
    arch: str,
    init: str,
    gs_level: str,
    repeat_idx: int,
    xtr: np.ndarray, ytr: np.ndarray,
    xva: np.ndarray, yva: np.ndarray,
    xte: np.ndarray, yte: np.ndarray,
    mean: float, std: float,
) -> Tuple[Dict[str, object], nn.Module]:
    """
    One repeat = one full train/val/test run.
    Returns (metrics, model).
    """
    # Repeat seed (stable)
    repeat_seed = GLOBAL_SEED + 1000 * repeat_idx
    set_global_seed(repeat_seed)

    dl_gen = torch.Generator()
    dl_gen.manual_seed(repeat_seed)

    model = get_model(arch, init, NUM_CLASSES)

    train_ds = AdaptiveNormDataset(xtr, ytr, mean, std, augment=True)
    val_ds   = AdaptiveNormDataset(xva, yva, mean, std, augment=False)
    test_ds  = AdaptiveNormDataset(xte, yte, mean, std, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=dl_gen
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=dl_gen
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=2, pin_memory=True,
        worker_init_fn=seed_worker, generator=dl_gen
    )

    # Optim (matches original AdamW style)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    # Simple cosine schedule (close to original spirit)
    steps_per_epoch = max(1, len(train_loader))
    total_steps = NUM_EPOCHS * steps_per_epoch
    warmup_steps = int(0.10 * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    best_val = -1.0
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    patience = 8
    patience_ctr = 0
    min_epochs = 0

    for epoch in range(NUM_EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        # val acc
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                pred = model(xb).argmax(dim=1)
                correct += int((pred == yb).sum().item())
                total += int(yb.numel())
        val_acc = 100.0 * correct / max(total, 1)

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= patience and epoch >= min_epochs:
            break

    model.load_state_dict(best_state)
    metrics = evaluate_model(model, test_loader)

    return metrics, model


print("✅ Cell 7 complete.")


✅ Cell 7 complete.


In [8]:
# =============================================================================
# CELL 8: Saving & Aggregation Helpers
# =============================================================================

RESULTS_DATA_DIR = OASIS_RESULTS_ROOT
RESULTS_DATA_DIR.mkdir(parents=True, exist_ok=True)


def level_to_filename(gs_level: str) -> str:
    """
    Converts GS level to filename-safe token.
    Examples:
      "Raw"    -> "RAW"
      "GS 0%"  -> "GS0"
      "GS 10%" -> "GS10"
    """
    if gs_level == "Raw":
        return "RAW"
    return f"GS{int(round(GS_PERCENTAGE_MAP[gs_level]))}"


def model_save_path(arch: str, init: str, gs_level: str) -> Path:
    """
    CLEAN model naming (no training fraction).
    
    Example outputs:
      resnet18_Pretrained_RAW_best.pth
      resnet18_Pretrained_GS10_best.pth
      densenet121_Scratch_GS50_best.pth
    """
    name = f"{arch}_{init}_{level_to_filename(gs_level)}_best.pth"
    out_dir = OASIS_MODELS_ROOT / arch
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / name


def ci95(mean: float, std: float, n: int):
    if n <= 1:
        return mean, mean
    sem = std / math.sqrt(n)
    h = 1.96 * sem
    return mean - h, mean + h


def run_group_id(arch: str, init: str) -> str:
    """
    Keep RunGroupID descriptive but NOT tied to filenames.
    This is metadata only.
    """
    return (
        f"OASIS__{arch}_{init}__"
        f"NR{NUM_REPEATS}__E{NUM_EPOCHS}__LR{LEARNING_RATE}__"
        f"GSit{GS_ITERATIONS}__B{BATCH_SIZE}"
    )


def save_master_csv(df: pd.DataFrame) -> Path:
    """
    Single authoritative CSV produced by this notebook.
    """
    out = RESULTS_DATA_DIR / "OASIS_Complete_Results.csv"
    df.to_csv(out, index=False)
    return out


def save_per_model_csv(df: pd.DataFrame, arch: str, init: str) -> Path:
    """
    Per-architecture + initialization CSVs (matches your originals).
    """
    out = RESULTS_DATA_DIR / f"OASIS_{arch}_{init}_results.csv"
    df_sub = df[(df["Model"] == arch) & (df["Initialization"] == init)]
    df_sub.to_csv(out, index=False)
    return out

def model_exists(arch: str, init: str, gs_level: str) -> bool:
    return model_save_path(arch, init, gs_level).exists()


def csv_row_exists(df: pd.DataFrame, arch: str, init: str, gs_level: str) -> bool:
    if df.empty:
        return False
    return (
        (df["Model"] == arch) &
        (df["Initialization"] == init) &
        (df["GS_Level"] == gs_level)
    ).any()



print("✅ Cell 8 complete.")


✅ Cell 8 complete.


In [10]:
# =============================================================================
# CELL 9: Main Execution (RESUMABLE)
# =============================================================================

# -------------------------------------------------------------------------
# Load existing results if resuming
# -------------------------------------------------------------------------

existing_rows = []
existing_csv = OASIS_RESULTS_ROOT / "OASIS_Complete_Results.csv"

if RESUME_IF_EXISTS and existing_csv.exists():
    df_existing = pd.read_csv(existing_csv)
    existing_rows = df_existing.to_dict("records")
    print(f"🔄 Loaded {len(existing_rows)} existing result rows")
else:
    df_existing = pd.DataFrame()
    print("🆕 No existing results found — starting fresh")

# IMPORTANT: preserve existing rows
all_rows: List[Dict[str, object]] = list(existing_rows)

print("\n" + "="*100)
print("🚀 OASIS REID (PID) — SINGLE SPLIT + REPEATS")
print(f"Time: {now_str()}")
print("="*100)

# -------------------------------------------------------------------------
# Main experiment loop
# -------------------------------------------------------------------------

for arch in ARCHITECTURES:
    for init in INITIALIZATIONS:

        rgid = run_group_id(arch, init)

        print("\n" + "█"*90)
        print(f"EXPERIMENT: {arch} | {init} | RunGroupID={rgid}")
        print("█"*90)

        raw_perrun_top1 = None  # needed for paired stats

        for gs_level in GS_LEVELS:

            # -------------------------------------------------------------
            # Resume guard: skip GS condition if already in CSV
            # -------------------------------------------------------------
            if RESUME_IF_EXISTS and csv_row_exists(df_existing, arch, init, gs_level):
                print(f"⏭️  Skipping {arch} | {init} | {gs_level} (already computed)")
                continue

            print("\n" + "-"*80)
            print(f"Condition: {gs_level}")
            print("-"*80)

            # -------------------------------------------------------------
            # Load or build data for this GS condition
            # -------------------------------------------------------------
            xtr, xva, xte, mean, std, pairm = load_or_build_condition_arrays(gs_level)

            perrun_top1 = []
            perrun_top5 = []
            perrun_f1 = []
            perrun_tp5m = []
            perrun_tp5med = []

            saved_model = None
            model_already_exists = RESUME_IF_EXISTS and model_exists(arch, init, gs_level)

            # -------------------------------------------------------------
            # Repeats
            # -------------------------------------------------------------
            for r in range(NUM_REPEATS):

                if model_already_exists:
                    if r == 0:
                        print(f"⏭️  Using existing model for {arch} | {init} | {gs_level}")
                    continue

                m, model_obj = train_one_repeat(
                    arch=arch, init=init, gs_level=gs_level, repeat_idx=r,
                    xtr=xtr, ytr=y_train,
                    xva=xva, yva=y_val,
                    xte=xte, yte=y_test,
                    mean=mean, std=std
                )

                perrun_top1.append(float(m["top1_acc"]))
                perrun_top5.append(float(m["top5_acc"]))
                perrun_f1.append(float(m["f1_macro"]))
                perrun_tp5m.append(float(m["trueprob_top5_mean"]))
                perrun_tp5med.append(float(m["trueprob_top5_median"]))

                if r == 0:
                    saved_model = model_obj

                print(
                    f"  Repeat {r+1}/{NUM_REPEATS}: "
                    f"Top1={m['top1_acc']:.2f}% | "
                    f"Top5={m['top5_acc']:.2f}% | "
                    f"F1={m['f1_macro']:.4f}"
                )

            # -------------------------------------------------------------
            # If model existed, load metrics from CSV (no retraining)
            # -------------------------------------------------------------
            if model_already_exists:
                row = df_existing[
                    (df_existing["Model"] == arch) &
                    (df_existing["Initialization"] == init) &
                    (df_existing["GS_Level"] == gs_level)
                ].iloc[0].to_dict()

                all_rows.append(row)
                continue

            # -------------------------------------------------------------
            # Aggregate metrics
            # -------------------------------------------------------------
            top1_mean = float(np.mean(perrun_top1))
            top1_std = float(np.std(perrun_top1, ddof=1)) if len(perrun_top1) > 1 else 0.0
            top1_ci_lo, top1_ci_hi = ci95(top1_mean, top1_std, len(perrun_top1))

            top5_mean = float(np.mean(perrun_top5))
            top5_std = float(np.std(perrun_top5, ddof=1)) if len(perrun_top5) > 1 else 0.0

            f1_mean = float(np.mean(perrun_f1))
            f1_std = float(np.std(perrun_f1, ddof=1)) if len(perrun_f1) > 1 else 0.0

            tp5_mean = float(np.mean(perrun_tp5m))
            tp5_std = float(np.std(perrun_tp5m, ddof=1)) if len(perrun_tp5m) > 1 else 0.0
            tp5_med = float(np.median(perrun_tp5med))

            # -------------------------------------------------------------
            # Save model (repeat 0 only)
            # -------------------------------------------------------------
            if saved_model is not None:
                sp = model_save_path(arch, init, gs_level)
                torch.save(
                    {
                        "arch": arch,
                        "init": init,
                        "gs_level": gs_level,
                        "training_fraction": TRAINING_FRACTION,
                        "num_classes": NUM_CLASSES,
                        "state_dict": saved_model.state_dict(),
                        "top1_mean": top1_mean,
                        "perrun_top1": perrun_top1,
                    },
                    sp
                )
                print(f"💾 Saved model: {sp.name}")

            if gs_level == "Raw":
                raw_perrun_top1 = perrun_top1[:]

            acc_drop_pp = ""
            acc_drop_rel = ""
            p_value = ""

            if raw_perrun_top1 is not None and gs_level != "Raw":
                acc_drop_pp = float(np.mean(raw_perrun_top1) - top1_mean)
                acc_drop_rel = float(acc_drop_pp / (np.mean(raw_perrun_top1) + 1e-8) * 100.0)
                if len(raw_perrun_top1) == len(perrun_top1):
                    _, p_value = stats.ttest_rel(raw_perrun_top1, perrun_top1)

            row = dict(
                RunGroupID=rgid,
                Model=arch,
                Initialization=init,
                GS_Level=gs_level,
                GS_Percentage=float(GS_PERCENTAGE_MAP[gs_level]),
                TrainingFraction=float(TRAINING_FRACTION),

                Top1_Mean=top1_mean,
                Top1_Std=top1_std,
                Top1_CI_Lower=float(top1_ci_lo),
                Top1_CI_Upper=float(top1_ci_hi),

                Top5_Mean=top5_mean,
                Top5_Std=top5_std,

                F1_Mean=f1_mean,
                F1_Std=f1_std,

                TrueProbTop5_Mean=tp5_mean,
                TrueProbTop5_Std=tp5_std,
                TrueProbTop5_Median=tp5_med,

                LPIPS=float(pairm.get("LPIPS", np.nan)),
                MutualInfo=float(pairm.get("MutualInfo", np.nan)),
                Entropy_GS=float(pairm.get("Entropy_GS", np.nan)),
                Entropy_Raw=float(pairm.get("Entropy_Raw", np.nan)),

                N_Repeats=int(NUM_REPEATS),
                PerRun_Top1=str(perrun_top1),

                AccuracyDrop_pp=acc_drop_pp,
                AccuracyDrop_relative=acc_drop_rel,
                pvalue_vs_raw_paired=p_value,
            )

            all_rows.append(row)

        # Save per-model CSV
        df_tmp = pd.DataFrame(all_rows)
        pm_csv = save_per_model_csv(df_tmp, arch, init)
        print(f"\n📄 Saved per-model CSV: {pm_csv.name}")

# -------------------------------------------------------------------------
# Final save
# -------------------------------------------------------------------------

df_final = pd.DataFrame(all_rows)
out_csv = save_master_csv(df_final)

print("\n" + "="*100)
print("✅ DONE")
print(f"Master CSV: {out_csv}")
print("="*100)

df_final.head(10)


🔄 Loaded 70 existing result rows

🚀 OASIS REID (PID) — SINGLE SPLIT + REPEATS
Time: 2026-02-12 04:51:39

██████████████████████████████████████████████████████████████████████████████████████████
EXPERIMENT: resnet18 | Pretrained | RunGroupID=OASIS__resnet18_Pretrained__NR5__E30__LR0.0005__GSit50__B16
██████████████████████████████████████████████████████████████████████████████████████████
⏭️  Skipping resnet18 | Pretrained | Raw (already computed)
⏭️  Skipping resnet18 | Pretrained | GS 0% (already computed)
⏭️  Skipping resnet18 | Pretrained | GS 10% (already computed)
⏭️  Skipping resnet18 | Pretrained | GS 20% (already computed)
⏭️  Skipping resnet18 | Pretrained | GS 30% (already computed)
⏭️  Skipping resnet18 | Pretrained | GS 40% (already computed)
⏭️  Skipping resnet18 | Pretrained | GS 50% (already computed)

📄 Saved per-model CSV: OASIS_resnet18_Pretrained_results.csv

██████████████████████████████████████████████████████████████████████████████████████████
EXPERIMENT: res

,RunGroupID,Model,Initialization,GS_Level,GS_Percentage,TrainingFraction,Top1_Mean,Top1_Std,Top1_CI_Lower,Top1_CI_Upper,...,N_Repeats,PerRun_Top1,Top1_GSTrain_RawTest,Top5_GSTrain_RawTest,F1_GSTrain_RawTest,TrueProbTop5Mean_GSTrain_RawTest,TrueProbTop5Median_GSTrain_RawTest,AccuracyDrop_pp,AccuracyDrop_relative,pvalue_vs_raw_paired
0,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,Raw,0.0,1.0,83.861671,8.958731,72.737938,94.985405,...,5,"[89.72142170989433, 72.8146013448607, 90.97022...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 0%,0.0,1.0,80.384246,9.553748,68.521702,92.246790,...,5,"[86.55139289145053, 67.14697406340058, 88.2804...",14.409222,30.451489,0.110536,0.024377,0.0,3.477426,4.146621,0.004683
2,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 10%,10.0,1.0,72.161383,11.336964,58.084683,86.238083,...,5,"[81.17195004803074, 59.5581171950048, 78.67435...",0.576369,2.017291,0.000155,0.004480,0.0,11.700288,13.951890,0.000820
3,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 20%,20.0,1.0,68.414986,11.733744,53.845619,82.984353,...,5,"[76.84918347742556, 54.947166186359276, 76.849...",0.384246,2.209414,0.000152,0.003967,0.0,15.446686,18.419244,0.000285
4,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 30%,30.0,1.0,63.650336,11.563265,49.292647,78.008026,...,5,"[72.8146013448607, 50.14409221902017, 70.31700...",0.096061,1.825168,0.000006,0.001539,0.0,20.211335,24.100802,0.000110
5,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 40%,40.0,1.0,59.135447,9.818639,46.943996,71.326898,...,5,"[65.51392891450529, 49.183477425552354, 66.186...",0.864553,2.497598,0.000186,0.004291,0.0,24.726225,29.484536,0.000007
6,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 50%,50.0,1.0,52.756964,11.308236,38.715935,66.797993,...,5,"[61.67146974063401, 41.210374639769455, 60.614...",0.576369,2.305476,0.000186,0.003481,0.0,31.104707,37.090493,0.000019
7,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,Raw,0.0,1.0,83.861671,8.958731,72.737938,94.985405,...,5,"[89.72142170989433, 72.8146013448607, 90.97022...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 0%,0.0,1.0,80.384246,9.553748,68.521702,92.246790,...,5,"[86.55139289145053, 67.14697406340058, 88.2804...",14.409222,30.451489,0.110536,0.024377,0.0,3.477426,4.146621,0.004683
9,OASIS__resnet18_Pretrained__NR5__E30__LR0.0005...,resnet18,Pretrained,GS 10%,10.0,1.0,72.161383,11.336964,58.084683,86.238083,...,5,"[81.17195004803074, 59.5581171950048, 78.67435...",0.576369,2.017291,0.000155,0.004480,0.0,11.700288,13.951890,0.000820
